# Memory AI Lab — 07 : Diagnostic sur-segmentation

**GPU recommandé** mais CPU possible (pas de training).

## Objectif

Le pipeline V4 B+C produit **321 épisodes** pour **193 gold** sur le test set.  
Ce notebook identifie **pourquoi** sans lire le contenu des messages.

## 3 hypothèses à tester

```
H1 — Micro-épisodes    : les faux épisodes sont très petits (1-3 msgs)
H2 — Artefacts temporels : les fausses frontières surviennent à faibles gaps
H3 — Bifurcations      : les faux épisodes adjacents partagent le même espace sémantique
```

## Méthode

```
Frontières prédites ∩ gold    → vrais positifs  (TP)
Frontières prédites \ gold    → faux positifs   (FP) ← sur-segmentation
Frontières gold \ prédites    → faux négatifs   (FN) ← sous-segmentation

Pour chaque FP : mesurer taille, gap, similarité, overlap entités
Comparer les distributions FP vs TP → quelle hypothèse est supportée ?
```

## Données requises
```
group_anon.txt
group_gold_test.json
group_embeddings_me5.npy
boundary_detector_tcn.pt   (ou boundary_detector.pt)
```

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
print('✓ OK')

In [ ]:
# ── CELLULE 3 : Google Drive + Copie locale ────────────────────────────────
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

for fname in ['group_anon.txt', 'group_gold_test.json',
              'group_embeddings_me5.npy',
              'boundary_detector_tcn.pt', 'boundary_detector.pt']:
    src, dst = f'{DRIVE_DIR}/{fname}', f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  Copié : {fname} ✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  ⚠️  {fname} absent (ignoré)')

DATA_DIR = LOCAL_DIR
print(f'\n✓ DATA_DIR = {DATA_DIR}')

In [ ]:
# ── CELLULE 4 : Charger données + pipeline V4 B+C ─────────────────────────
import numpy as np
import json
import torch
from pathlib import Path
from parsers.whatsapp_parser import parse_whatsapp_chat
from episode_segmenter_hybrid import HybridEpisodeSegmenter
from episode_algorithm_fast import EpisodeSegmenterFast
from episode_merger import EpisodeMerger
from episode_resegmenter_fast import EpisodeResegmenterFast

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

# ── Artefacts + embeddings ────────────────────────────────────────────────
all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
all_embeddings = np.load(f'{DATA_DIR}/group_embeddings_me5.npy')
print(f'Total artefacts : {len(all_artifacts)}')

# ── Gold test ─────────────────────────────────────────────────────────────
with open(f'{DATA_DIR}/group_gold_test.json') as f:
    gold_data = json.load(f)

gold_episodes = gold_data['episodes']
n_test = len(gold_data['artifacts'])
n_tune = len(all_artifacts) - n_test  # test set = fin du corpus

test_artifacts  = all_artifacts[n_tune:n_tune + n_test]
test_embeddings = all_embeddings[n_tune:n_tune + n_test]

# Labels gold → y_true
y_true = [None] * n_test
gold_boundaries = set()  # indices où commence un nouvel épisode
for ep in gold_episodes:
    for idx in range(ep['start_idx'], ep['end_idx'] + 1):
        if idx < n_test:
            y_true[idx] = ep['episode_id']
    if ep['start_idx'] > 0:
        gold_boundaries.add(ep['start_idx'])

print(f'Test set        : {n_test} msgs · {len(gold_episodes)} épisodes gold')
print(f'n_tune (offset) : {n_tune}')
print(f'Frontières gold : {len(gold_boundaries)}')

# ── Boundary detector ─────────────────────────────────────────────────────
DETECTOR = None
TCN_PATH = f'{DATA_DIR}/boundary_detector_tcn.pt'
MLP_PATH = f'{DATA_DIR}/boundary_detector.pt'

if os.path.exists(TCN_PATH):
    from boundary_detector_tcn import TCNBoundaryDetector
    DETECTOR = TCNBoundaryDetector(device=device).load(TCN_PATH)
    print(f'✓ TCN boundary detector (seuil={DETECTOR.threshold:.3f})')
elif os.path.exists(MLP_PATH):
    from boundary_detector import BoundaryDetector
    DETECTOR = BoundaryDetector(device=device)
    DETECTOR.load(MLP_PATH)
    print(f'✓ MLP boundary detector (seuil={DETECTOR.threshold:.3f})')
else:
    print('⚠️  Pas de boundary detector — mode fast uniquement')

# ── Paramètres V4 B+C ─────────────────────────────────────────────────────
PARAMS_V4 = dict(
    attach_threshold       = 0.434,
    ema_alpha              = 0.787,
    time_threshold_minutes = 330,
    boundary_k             = 0.107,
    alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
    dormancy_minutes       = 1440,
    hard_break_minutes     = 0,
    active_penalty_hours   = 24.0,
    allow_reactivation     = True,
)
MERGER_PARAMS = dict(min_size=1, merge_sim=0.877, merge_gap_minutes=60.)
FIXED_RESEG   = dict(max_iter=3, window_minutes=480., min_ep_size=2)

print('✓ Paramètres V4 B+C chargés')

In [ ]:
# ── CELLULE 5 : Segmenter + extraire les frontières prédites ──────────────
from sklearn.metrics import adjusted_rand_score

seg_params = {k: v for k, v in PARAMS_V4.items()}

if DETECTOR is not None:
    seg = HybridEpisodeSegmenter(detector=DETECTOR, device=device, **seg_params)
else:
    fast_keys = {'attach_threshold','ema_alpha','time_threshold_minutes',
                 'alpha','beta','gamma','delta','rho',
                 'dormancy_minutes','hard_break_minutes',
                 'active_penalty_hours','allow_reactivation'}
    seg = EpisodeSegmenterFast(device=device, **{k:v for k,v in seg_params.items() if k in fast_keys})

pred_episodes_raw = seg.consolidate(seg.segment(test_artifacts, test_embeddings))

merger = EpisodeMerger(device=device, **MERGER_PARAMS)
pred_episodes_merged = merger.merge(pred_episodes_raw)

reseg = EpisodeResegmenterFast(**FIXED_RESEG)
pred_episodes = reseg.resegment(pred_episodes_merged, test_artifacts, test_embeddings, seg)

# Frontières prédites
pred_boundaries = set()
for ep in pred_episodes:
    if ep.artifact_indices:
        start = min(ep.artifact_indices)
        if start > 0:
            pred_boundaries.add(start)

# ARI
y_pred = [None] * n_test
for ep in pred_episodes:
    for idx in ep.artifact_indices:
        if idx < n_test:
            y_pred[idx] = ep.id

pairs = [(t, p) for t, p in zip(y_true, y_pred) if t is not None and p is not None]
yt, yp = zip(*pairs)
ari = adjusted_rand_score(yt, yp)

# Analyse erreurs
TP = pred_boundaries & gold_boundaries
FP = pred_boundaries - gold_boundaries   # sur-segmentation
FN = gold_boundaries - pred_boundaries   # sous-segmentation

precision = len(TP) / len(pred_boundaries) if pred_boundaries else 0
recall    = len(TP) / len(gold_boundaries)  if gold_boundaries else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'=== Pipeline V4 B+C sur TEST ===')
print(f'ARI           : {ari:+.4f}')
print(f'Épisodes gold : {len(gold_episodes)}')
print(f'Épisodes pred : {len(pred_episodes)}')
print()
print(f'=== Analyse frontières ===')
print(f'TP (correct)  : {len(TP):4d}  frontières bien placées')
print(f'FP (faux +)   : {len(FP):4d}  frontières en trop  ← sur-segmentation')
print(f'FN (faux -)   : {len(FN):4d}  frontières manquées ← sous-segmentation')
print(f'Precision     : {precision:.3f}')
print(f'Recall        : {recall:.3f}')
print(f'F1            : {f1:.3f}')

In [ ]:
# ── CELLULE 6 : Construire les features par frontière ─────────────────────
# Pour chaque frontière (TP ou FP), calculer 5 signaux
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_episode_centroid(episode, embeddings, n_test):
    """Centroïd moyen des embeddings de l'épisode."""
    valid = [i for i in episode.artifact_indices if i < n_test]
    if not valid:
        return None
    return embeddings[valid].mean(axis=0)

# Map : position → épisode
pos_to_episode = {}
for ep in pred_episodes:
    for idx in ep.artifact_indices:
        pos_to_episode[idx] = ep

def get_temporal_gap_minutes(pos, artifacts):
    """Gap en minutes entre message pos-1 et pos."""
    if pos == 0 or pos >= len(artifacts):
        return 0.0
    t1 = artifacts[pos - 1].timestamp
    t2 = artifacts[pos].timestamp
    if t1 is None or t2 is None:
        return 0.0
    return max(0.0, (t2 - t1).total_seconds() / 60)

def get_semantic_similarity(pos, embeddings):
    """Cosine similarity entre msg[pos] et msg[pos-1]."""
    if pos == 0 or pos >= len(embeddings):
        return 1.0
    a = embeddings[pos - 1].reshape(1, -1)
    b = embeddings[pos].reshape(1, -1)
    return float(cosine_similarity(a, b)[0, 0])

def get_centroid_similarity(pos, embeddings, pos_to_episode, n_test):
    """Cosine similarity entre msg[pos] et centroïd de l'épisode précédent."""
    if pos == 0:
        return 1.0
    prev_ep = pos_to_episode.get(pos - 1)
    if prev_ep is None:
        return 1.0
    centroid = get_episode_centroid(prev_ep, embeddings, n_test)
    if centroid is None:
        return 1.0
    a = centroid.reshape(1, -1)
    b = embeddings[pos].reshape(1, -1)
    return float(cosine_similarity(a, b)[0, 0])

def get_episode_size_after(pos, pos_to_episode):
    """Taille de l'épisode créé après la frontière."""
    ep = pos_to_episode.get(pos)
    if ep is None:
        return 0
    return len(ep.artifact_indices)

def get_episode_size_before(pos, pos_to_episode):
    """Taille de l'épisode avant la frontière."""
    ep = pos_to_episode.get(pos - 1) if pos > 0 else None
    if ep is None:
        return 0
    return len(ep.artifact_indices)

def get_episode_duration_minutes(pos, pos_to_episode, artifacts):
    """Durée en minutes de l'épisode créé après la frontière."""
    ep = pos_to_episode.get(pos)
    if ep is None or len(ep.artifact_indices) < 2:
        return 0.0
    indices = sorted(ep.artifact_indices)
    t1 = artifacts[indices[0]].timestamp
    t2 = artifacts[indices[-1]].timestamp
    if t1 is None or t2 is None:
        return 0.0
    return max(0.0, (t2 - t1).total_seconds() / 60)

# Calculer les features pour TP et FP
def compute_features(boundary_set, label):
    rows = []
    for pos in sorted(boundary_set):
        rows.append({
            'pos':              pos,
            'label':            label,
            'gap_minutes':      get_temporal_gap_minutes(pos, test_artifacts),
            'sim_last_msg':     get_semantic_similarity(pos, test_embeddings),
            'sim_centroid':     get_centroid_similarity(pos, test_embeddings, pos_to_episode, n_test),
            'size_after':       get_episode_size_after(pos, pos_to_episode),
            'size_before':      get_episode_size_before(pos, pos_to_episode),
            'duration_minutes': get_episode_duration_minutes(pos, pos_to_episode, test_artifacts),
        })
    return rows

features_TP = compute_features(TP, 'TP')
features_FP = compute_features(FP, 'FP')
all_features = features_TP + features_FP

print(f'✓ Features calculées : {len(features_TP)} TP · {len(features_FP)} FP')

In [ ]:
# ── CELLULE 7 : H1 — Micro-épisodes ──────────────────────────────────────
# Hypothèse : les faux épisodes (FP) sont beaucoup plus petits que les vrais (TP)
import numpy as np

tp_sizes = [r['size_after'] for r in features_TP]
fp_sizes = [r['size_after'] for r in features_FP]

print('=== H1 : Taille des épisodes créés à chaque frontière ===')
print(f'                   TP (correct)    FP (faux +)')
print(f'  Médiane        : {np.median(tp_sizes):8.1f}        {np.median(fp_sizes):8.1f}')
print(f'  Moyenne        : {np.mean(tp_sizes):8.1f}        {np.mean(fp_sizes):8.1f}')
print(f'  Min            : {np.min(tp_sizes):8.0f}        {np.min(fp_sizes):8.0f}')
print(f'  Max            : {np.max(tp_sizes):8.0f}        {np.max(fp_sizes):8.0f}')
print(f'  % épisodes ≤3  : {100*np.mean(np.array(tp_sizes)<=3):7.1f}%       {100*np.mean(np.array(fp_sizes)<=3):7.1f}%')
print(f'  % épisodes ≤5  : {100*np.mean(np.array(tp_sizes)<=5):7.1f}%       {100*np.mean(np.array(fp_sizes)<=5):7.1f}%')
print()

# Distribution par bins
bins = [1, 2, 3, 5, 8, 12, 20, 100]
print('  Distribution taille épisode après frontière :')
print(f'  {"Bin":15s}  {"TP":>8s}  {"FP":>8s}')
for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    tp_n = sum(1 for s in tp_sizes if lo <= s < hi)
    fp_n = sum(1 for s in fp_sizes if lo <= s < hi)
    bar_tp = '▓' * int(tp_n / max(len(tp_sizes), 1) * 20)
    bar_fp = '░' * int(fp_n / max(len(fp_sizes), 1) * 20)
    print(f'  [{lo:3d} - {hi:3d}[   {tp_n:4d} {bar_tp:<20s}  {fp_n:4d} {bar_fp}')

ratio = np.median(fp_sizes) / max(np.median(tp_sizes), 1)
h1_supported = ratio < 0.5
print(f'\n  → Ratio médiane FP/TP : {ratio:.2f}')
print(f'  → H1 micro-épisodes : {"SUPPORTÉE ✓" if h1_supported else "faiblement supportée"}')

In [ ]:
# ── CELLULE 8 : H2 — Artefacts temporels ─────────────────────────────────
# Hypothèse : les fausses frontières surviennent à de faibles gaps temporels

tp_gaps = [r['gap_minutes'] for r in features_TP]
fp_gaps = [r['gap_minutes'] for r in features_FP]

print('=== H2 : Gap temporel à la frontière (minutes) ===')
print(f'                   TP (correct)    FP (faux +)')
print(f'  Médiane        : {np.median(tp_gaps):8.1f}        {np.median(fp_gaps):8.1f}')
print(f'  Moyenne        : {np.mean(tp_gaps):8.1f}        {np.mean(fp_gaps):8.1f}')
print(f'  % gap < 30min  : {100*np.mean(np.array(tp_gaps)<30):7.1f}%       {100*np.mean(np.array(fp_gaps)<30):7.1f}%')
print(f'  % gap < 60min  : {100*np.mean(np.array(tp_gaps)<60):7.1f}%       {100*np.mean(np.array(fp_gaps)<60):7.1f}%')
print(f'  % gap > 6h     : {100*np.mean(np.array(tp_gaps)>360):7.1f}%       {100*np.mean(np.array(fp_gaps)>360):7.1f}%')
print()

# Distribution par bins temporels
time_bins = [0, 5, 15, 30, 60, 120, 360, 1440, 99999]
time_labels = ['<5min', '5-15min', '15-30min', '30-60min', '1-2h', '2-6h', '6-24h', '>24h']
print('  Distribution gap temporel :')
print(f'  {"Bin":12s}  {"TP":>8s}  {"FP":>8s}')
for i, lbl in enumerate(time_labels):
    lo, hi = time_bins[i], time_bins[i+1]
    tp_n = sum(1 for g in tp_gaps if lo <= g < hi)
    fp_n = sum(1 for g in fp_gaps if lo <= g < hi)
    bar_tp = '▓' * int(tp_n / max(len(tp_gaps), 1) * 20)
    bar_fp = '░' * int(fp_n / max(len(fp_gaps), 1) * 20)
    print(f'  {lbl:12s}  {tp_n:4d} {bar_tp:<20s}  {fp_n:4d} {bar_fp}')

h2_supported = np.median(fp_gaps) < np.median(tp_gaps) * 0.5
print(f'\n  → Médiane FP gap : {np.median(fp_gaps):.1f}min  vs  TP gap : {np.median(tp_gaps):.1f}min')
print(f'  → H2 artefact temporel : {"SUPPORTÉE ✓" if h2_supported else "faiblement supportée"}')

In [ ]:
# ── CELLULE 9 : H3 — Bifurcations sémantiques ────────────────────────────
# Hypothèse : les fausses frontières coupent dans un espace sémantique continu
# → sim(msg, centroïd épisode précédent) reste élevée après FP

tp_sim_msg     = [r['sim_last_msg']  for r in features_TP]
fp_sim_msg     = [r['sim_last_msg']  for r in features_FP]
tp_sim_centroid = [r['sim_centroid'] for r in features_TP]
fp_sim_centroid = [r['sim_centroid'] for r in features_FP]

print('=== H3 : Similarité sémantique à la frontière ===')
print()
print('  A. Similarité msg[t] vs msg[t-1] :')
print(f'                   TP (correct)    FP (faux +)')
print(f'  Médiane        : {np.median(tp_sim_msg):8.4f}        {np.median(fp_sim_msg):8.4f}')
print(f'  Moyenne        : {np.mean(tp_sim_msg):8.4f}        {np.mean(fp_sim_msg):8.4f}')
print(f'  % sim > 0.80   : {100*np.mean(np.array(tp_sim_msg)>0.80):7.1f}%       {100*np.mean(np.array(fp_sim_msg)>0.80):7.1f}%')
print(f'  % sim > 0.85   : {100*np.mean(np.array(tp_sim_msg)>0.85):7.1f}%       {100*np.mean(np.array(fp_sim_msg)>0.85):7.1f}%')
print()
print('  B. Similarité msg[t] vs centroïd épisode précédent :')
print(f'                   TP (correct)    FP (faux +)')
print(f'  Médiane        : {np.median(tp_sim_centroid):8.4f}        {np.median(fp_sim_centroid):8.4f}')
print(f'  Moyenne        : {np.mean(tp_sim_centroid):8.4f}        {np.mean(fp_sim_centroid):8.4f}')
print(f'  % sim > 0.80   : {100*np.mean(np.array(tp_sim_centroid)>0.80):7.1f}%       {100*np.mean(np.array(fp_sim_centroid)>0.80):7.1f}%')

h3_supported = np.median(fp_sim_centroid) > np.median(tp_sim_centroid) + 0.05
print(f'\n  → Médiane sim centroïd FP : {np.median(fp_sim_centroid):.4f}')
print(f'    Médiane sim centroïd TP : {np.median(tp_sim_centroid):.4f}')
print(f'  → H3 bifurcations : {"SUPPORTÉE ✓" if h3_supported else "faiblement supportée"}')

In [ ]:
# ── CELLULE 10 : Durée des faux épisodes ─────────────────────────────────
# Signal supplémentaire : les épisodes produits par FP sont-ils courts ?

tp_dur = [r['duration_minutes'] for r in features_TP]
fp_dur = [r['duration_minutes'] for r in features_FP]

print('=== Durée des épisodes créés (minutes) ===')
print(f'                   TP (correct)    FP (faux +)')
print(f'  Médiane        : {np.median(tp_dur):8.1f}        {np.median(fp_dur):8.1f}')
print(f'  Moyenne        : {np.mean(tp_dur):8.1f}        {np.mean(fp_dur):8.1f}')
print(f'  % < 30min      : {100*np.mean(np.array(tp_dur)<30):7.1f}%       {100*np.mean(np.array(fp_dur)<30):7.1f}%')
print(f'  % < 60min      : {100*np.mean(np.array(tp_dur)<60):7.1f}%       {100*np.mean(np.array(fp_dur)<60):7.1f}%')
print(f'  % < 10min      : {100*np.mean(np.array(tp_dur)<10):7.1f}%       {100*np.mean(np.array(fp_dur)<10):7.1f}%')

In [ ]:
# ── CELLULE 11 : Synthèse — quelle hypothèse est supportée ? ─────────────

print('╔══════════════════════════════════════════════════════════╗')
print('║  SYNTHÈSE DIAGNOSTIC SUR-SEGMENTATION                  ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  FP (frontières en trop)    : {len(FP):4d}                    ║')
print(f'║  FN (frontières manquées)   : {len(FN):4d}                    ║')
print(f'║  Precision boundary         : {precision:.3f}                   ║')
print(f'║  Recall    boundary         : {recall:.3f}                   ║')
print('╠══════════════════════════════════════════════════════════╣')

hypotheses = [
    ('H1 — Micro-épisodes', h1_supported,
     f'médiane taille FP={np.median(fp_sizes):.0f} vs TP={np.median(tp_sizes):.0f}',
     'Contrainte taille minimale + post-hoc merge par taille'),
    ('H2 — Artefacts temporels', h2_supported,
     f'médiane gap FP={np.median(fp_gaps):.0f}min vs TP={np.median(tp_gaps):.0f}min',
     'Augmenter poids γ·Temp ou seuil temporel minimal'),
    ('H3 — Bifurcations', h3_supported,
     f'sim centroïd FP={np.median(fp_sim_centroid):.3f} vs TP={np.median(tp_sim_centroid):.3f}',
     'Lookahead N messages + post-hoc merge sémantique'),
]

for name, supported, evidence, action in hypotheses:
    status = 'SUPPORTÉE ✓' if supported else 'non supportée'
    print(f'║                                                          ║')
    print(f'║  {name:<52} ║')
    print(f'║  → {status:<52} ║')
    print(f'║    {evidence:<52} ║')
    if supported:
        print(f'║    Action : {action:<44} ║')

print('╠══════════════════════════════════════════════════════════╣')
n_supported = sum(s for _, s, _, _ in hypotheses)
if n_supported == 0:
    conclusion = 'Aucune hypothèse claire → inspecter manuellement 10 FP'
elif n_supported == 1:
    conclusion = 'Une cause dominante → action ciblée'
else:
    conclusion = 'Causes multiples → combiner les corrections'
print(f'║  Conclusion : {conclusion:<42} ║')
print('╚══════════════════════════════════════════════════════════╝')

In [ ]:
# ── CELLULE 12 : Test rapide des corrections candidates ───────────────────
# Sans retraining — corrections post-hoc sur la segmentation existante

from sklearn.metrics import adjusted_rand_score

def ari_from_episodes(episodes, y_true, n):
    y_pred = [None] * n
    for ep in episodes:
        for idx in ep.artifact_indices:
            if idx < n:
                y_pred[idx] = ep.id
    pairs = [(t, p) for t, p in zip(y_true, y_pred) if t is not None and p is not None]
    if not pairs:
        return 0.0
    yt, yp = zip(*pairs)
    return adjusted_rand_score(yt, yp)

baseline_ari = ari_from_episodes(pred_episodes, y_true, n_test)
print(f'Baseline ARI (V4 B+C) : {baseline_ari:+.4f}  ({len(pred_episodes)} épisodes)')
print()

# ── Correction 1 : taille minimale ───────────────────────────────────────
print('Correction 1 — Taille minimale :')
for min_size in [2, 3, 4, 5]:
    # Fusionner les micro-épisodes avec l'épisode adjacent le plus similaire
    filtered = [ep for ep in pred_episodes if len(ep.artifact_indices) >= min_size]
    ari_filtered = ari_from_episodes(filtered, y_true, n_test)
    print(f'  min_size={min_size} : ARI={ari_filtered:+.4f}  ({len(filtered)} épisodes)')

print()

# ── Correction 2 : merge sémantique agressif ─────────────────────────────
print('Correction 2 — Merge sémantique (seuil similarity) :')
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

def post_merge(episodes, threshold, embeddings, n_test):
    """Fusionne les épisodes adjacents dont les centroïds sont très similaires."""
    eps = sorted(episodes, key=lambda e: min(e.artifact_indices) if e.artifact_indices else 0)
    merged = True
    while merged:
        merged = False
        new_eps = []
        i = 0
        while i < len(eps):
            if i + 1 < len(eps):
                ep1, ep2 = eps[i], eps[i+1]
                v1 = embeddings[[idx for idx in ep1.artifact_indices if idx < n_test]].mean(axis=0) if ep1.artifact_indices else None
                v2 = embeddings[[idx for idx in ep2.artifact_indices if idx < n_test]].mean(axis=0) if ep2.artifact_indices else None
                if v1 is not None and v2 is not None:
                    sim = float(cos_sim([v1], [v2])[0, 0])
                    if sim >= threshold:
                        # Fusionner
                        ep1.artifact_indices = sorted(ep1.artifact_indices + ep2.artifact_indices)
                        new_eps.append(ep1)
                        i += 2
                        merged = True
                        continue
            new_eps.append(eps[i])
            i += 1
        eps = new_eps
    return eps

for threshold in [0.88, 0.90, 0.92, 0.94]:
    merged_eps = post_merge(list(pred_episodes), threshold, test_embeddings, n_test)
    ari_merged = ari_from_episodes(merged_eps, y_true, n_test)
    print(f'  sim_threshold={threshold:.2f} : ARI={ari_merged:+.4f}  ({len(merged_eps)} épisodes)')

print()
print('→ La meilleure correction indique quelle hypothèse agir en priorité.')

## Lecture des résultats

### Si H1 supportée (micro-épisodes)
```
→ Ajouter contrainte min_size dans EpisodeMerger
→ Tuner min_size dans Optuna (cellule 7 de 01_eval_ari.ipynb)
→ Gain attendu : -20 à -40 épisodes
```

### Si H2 supportée (artefacts temporels)
```
→ Augmenter poids γ dans AttachScore (gamma > 0.10)
→ Ou ajouter un seuil temporel minimal avant de créer un épisode
→ Tuner gamma dans Optuna
```

### Si H3 supportée (bifurcations)
```
→ Implémenter lookahead N=3 messages
→ Ou post-hoc merge sémantique avec seuil élevé (0.90+)
→ Plus tard : classifier 3 classes (APPEND / BIFURCATION / NEW)
```

### Si aucune hypothèse claire
```
→ Exporter 10 FP dans un CSV anonymisé
→ Inspecter manuellement les positions et timestamps
```